In [1]:
from torch.utils.data import DataLoader, random_split
from utils.report import AverageMeter
from utils.metrics import calculate_accuracy

import os
import random
import torch
import warnings

import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings('ignore')

In [2]:
BATCH_SIZE = 256
LR = 0.001
ES_THRES = 5
SEED = 1234
OUTPUT_DIR = "./state_dicts/MNIST"

# Set Seed

In [3]:
def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

seed_everything(SEED)

# Reference

- https://dacon.io/competitions/official/235614/codeshare/1300
- https://github.com/bentrevett/pytorch-image-classification/blob/master/2_lenet.ipynb

# Download Datasets

In [4]:
from torchvision.datasets import MNIST
import torchvision.transforms as transforms

In [5]:
download_root = '../datasets/MNIST_DATASET'
train_data = MNIST(download_root, train=True, download=True)
mean = train_data.data.float().mean() / 255
std = train_data.data.float().std() / 255

print(f'Calculated mean: {mean}')
print(f'Calculated std: {std}')

Calculated mean: 0.13066047430038452
Calculated std: 0.30810779333114624


In [6]:
train_transforms = transforms.Compose([
    transforms.RandomRotation(5, fill=(0,)),
    transforms.RandomCrop(28, padding = 2),
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(32),
    transforms.ToTensor(),
#     transforms.Normalize(mean = [mean], std = [std])
])

test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(32),
    transforms.ToTensor(),
#     transforms.Normalize(mean = [mean], std = [std])
])

In [7]:
train_valid_dataset = MNIST(download_root, transform=train_transforms, train=True, download=True)
train_dataset, valid_dataset = random_split(train_valid_dataset, [54000, 6000])
test_dataset = MNIST(download_root, transform=test_transforms, train=False, download=True)

In [9]:
len(train_dataset)

54000

In [10]:
len(valid_dataset)

6000

# Models 

In [11]:
class FeatureExtractor(nn.Module):
    def __init__(self, in_channel_dim=3, out_channel_dim=16):
        super(FeatureExtractor, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel_dim, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=out_channel_dim, kernel_size=5)
        
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        
#         x = x.view(-1, 16 * 5 * 5)

        return x

In [12]:
class Classifier(nn.Module):
    def __init__(self, in_dim, out_dim=10):
        super(Classifier, self).__init__()
        self.fc1 = nn.Linear(in_dim, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, out_dim)
        
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        
        x = self.fc3(x)
        
        return x

In [13]:
class LeNet(nn.Module):
    def __init__(self, in_channel_dim=1, out_dim=10):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel_dim, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        
        self.fc1 = nn.Linear(256, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, out_dim)
        
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        
        x = torch.flatten(x, 1)
        
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        
        x = self.fc3(x)
        
        return x

# Train/Eval Functions

In [14]:
def train_loop_classifier(feature_extractor, classifier, train_loader, loss_func, optimizer, 
                          summary_loss, summary_acc=None, device=None):
    feature_extractor.train()
    classifier.train()
    
    for batch_idx, (data, target) in enumerate(train_loader):
        if device is not None:
            data, target = data.to(device), target.to(device)
            
        optimizer.zero_grad()
        feature = feature_extractor(data)
        feature = torch.flatten(feature, 1)
        output = classifier(feature)
        loss = loss_func(output, target)
        loss.backward()
        optimizer.step()
        
        summary_loss.update(loss.detach().item(), BATCH_SIZE)
        if summary_acc is not None:
            acc = calculate_accuracy(output, target)
            summary_acc.update(acc, BATCH_SIZE)
        
    return summary_loss, summary_acc

def eval_loop_classifier(feature_extractor, classifier, valid_loader, loss_func, optimizer, 
                         summary_loss, summary_acc=None, device=None):
    feature_extractor.eval()
    classifier.eval()
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(valid_loader):
            if device is not None:
                data, target = data.to(device), target.to(device)

            feature = feature_extractor(data)
            feature = torch.flatten(feature, 1)
            output = classifier(feature)

            loss = loss_func(output, target)

            summary_loss.update(loss.detach().item(), BATCH_SIZE)
            if summary_acc is not None:
                acc = calculate_accuracy(output, target)
                summary_acc.update(acc, BATCH_SIZE)

    return summary_loss, summary_acc

# Training with Early Stopping

In [15]:
def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    feature_extractor = FeatureExtractor()
    classifier = Classifier(400)

    feature_extractor.to(device)
    classifier.to(device)
    
    feature_extractor.train()
    classifier.train()

    criterion = nn.CrossEntropyLoss()
    criterion.to(device)
    
    model_parameters = list(feature_extractor.parameters()) + list(classifier.parameters())
    optimizer = optim.Adam(model_parameters, lr=LR)

    train_loader = DataLoader(dataset=train_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)

    valid_loader = DataLoader(dataset=valid_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)

    test_loader = DataLoader(dataset=test_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)

    best_epoch = None
    best_loss = None
    epoch = 0
    es_count = 0
    while(True):
        epoch += 1
        summary_loss_train = AverageMeter()
        summary_acc_train = AverageMeter()
        summary_loss_valid = AverageMeter()
        summary_acc_valid = AverageMeter()

        summary_loss_train, summary_acc_train = train_loop_classifier(feature_extractor, classifier, train_loader, 
                                                   criterion, optimizer, summary_loss_train, summary_acc_train, device=device)
        summary_loss_valid, summary_acc_valid = eval_loop_classifier(feature_extractor, classifier, valid_loader, 
                                                   criterion, optimizer, summary_loss_valid, summary_acc_valid, device=device)

        print(f"[epoch]{epoch} [train loss]{summary_loss_train.avg} [train acc]{summary_acc_train.avg}  [valid loss]{summary_loss_valid.avg} [valid acc]{summary_acc_valid.avg} ")

        if best_loss is None:
            best_loss = summary_loss_valid.avg
            best_epoch = epoch

        if best_loss > summary_loss_valid.avg:
            best_loss = summary_loss_valid.avg
            best_epoch = epoch
            es_count = 0
        else:
            es_count += 1

        if es_count == ES_THRES:
            break

    print(f"Best epoch: {best_epoch}")
    return best_epoch

In [16]:
best_epoch = run_training()

[epoch]1 [train loss]0.7395936635574458 [train acc]0.7689006328582764  [valid loss]0.27947041764855385 [valid acc]0.911435067653656 
[epoch]2 [train loss]0.20802747034489827 [train acc]0.9367495775222778  [valid loss]0.1768202514698108 [valid acc]0.9461263418197632 
[epoch]3 [train loss]0.15011772389801759 [train acc]0.953722357749939  [valid loss]0.1413039347777764 [valid acc]0.9571707248687744 
[epoch]4 [train loss]0.12296263313900803 [train acc]0.9623716473579407  [valid loss]0.12183039200802644 [valid acc]0.9627976417541504 
[epoch]5 [train loss]0.10520095593556408 [train acc]0.9680774211883545  [valid loss]0.10548159185176094 [valid acc]0.9667271375656128 
[epoch]6 [train loss]0.09177455500262609 [train acc]0.971002459526062  [valid loss]0.09960099356248975 [valid acc]0.9686105251312256 
[epoch]7 [train loss]0.08629274628703346 [train acc]0.973816454410553  [valid loss]0.07945416076108813 [valid acc]0.9772833585739136 
[epoch]8 [train loss]0.07740817573892561 [train acc]0.97557276

# Second Training

In [17]:
def run_second_training(best_epoch):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    feature_extractor = FeatureExtractor()
    classifier = Classifier(400)

    feature_extractor.to(device)
    classifier.to(device)
    
    feature_extractor.train()
    classifier.train()
    
    train_valid_loader = DataLoader(dataset=train_valid_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)

    criterion = nn.CrossEntropyLoss()
    model_parameters = list(feature_extractor.parameters()) + list(classifier.parameters())
    optimizer = optim.Adam(model_parameters, lr=LR)

    epoch = 0
    for _ in range(best_epoch):
        epoch += 1
        summary_loss_train = AverageMeter()
    #     summary_acc_train = AverageMeter()

        summary_loss_train, _ = train_loop_classifier(feature_extractor, classifier, train_valid_loader, 
                                                   criterion, optimizer, summary_loss_train, None, device=device)

        print(f"[epoch]{epoch} [train loss]{summary_loss_train.avg}")
    return feature_extractor, classifier

In [18]:
feature_extractor, classifier = run_second_training(best_epoch)

[epoch]1 [train loss]0.7633652446751898
[epoch]2 [train loss]0.20938260707449405
[epoch]3 [train loss]0.15230229056261954
[epoch]4 [train loss]0.12256177164455678
[epoch]5 [train loss]0.10444311998943065
[epoch]6 [train loss]0.09142235621334391
[epoch]7 [train loss]0.07891496413565696
[epoch]8 [train loss]0.07237939256778422
[epoch]9 [train loss]0.0649858702370461
[epoch]10 [train loss]0.06076496002521921
[epoch]11 [train loss]0.06219816083445194
[epoch]12 [train loss]0.05455004312890641
[epoch]13 [train loss]0.05260013139390565
[epoch]14 [train loss]0.04947870098688501
[epoch]15 [train loss]0.04848695677962709
[epoch]16 [train loss]0.044544452797383706
[epoch]17 [train loss]0.042760239077850856
[epoch]18 [train loss]0.04144823730705266
[epoch]19 [train loss]0.03991713569836414
[epoch]20 [train loss]0.038710914619584036
[epoch]21 [train loss]0.03925023035720942
[epoch]22 [train loss]0.037049986640031035
[epoch]23 [train loss]0.03612500192458801
[epoch]24 [train loss]0.03338291373183118

In [19]:
def run_test(feature_extractor, classifier):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    feature_extractor.eval()
    classifier.eval()
    
    test_loader = DataLoader(dataset=test_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)
    
    criterion = nn.CrossEntropyLoss()
    
    summary_loss_test = AverageMeter()
    summary_acc_test = AverageMeter()

    summary_loss_test, summary_acc_test = eval_loop_classifier(feature_extractor, classifier, test_loader, 
                                               criterion, None, summary_loss_test, summary_acc_test, device=device)

    print(f"[test loss]{summary_loss_test.avg} [test acc]{summary_acc_test.avg}")

In [20]:
run_test(feature_extractor, classifier)

[test loss]0.028741557616740465 [test acc]0.990917980670929


In [21]:
torch.save(feature_extractor.state_dict(), os.path.join(OUTPUT_DIR, f"eature_extractor.pth"))
torch.save(classifier.state_dict(), os.path.join(OUTPUT_DIR, "classifier.pth"))